In [101]:
from dotenv import load_dotenv
config = load_dotenv()

In [3]:
import asyncio
import base64
import difflib
import io
import json
import os
import re
import time

from PIL import Image
from anthropic import Anthropic, APIStatusError, APIError
from dotenv import load_dotenv
from jira import JIRA, JIRAError
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool, BaseTool
from prompt_toolkit import PromptSession
from prompt_toolkit.styles import Style
from pydantic import BaseModel
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.syntax import Syntax
from tavily import TavilyClient

In [102]:
class ModelConfig(BaseModel):
    provider: str = None
    model: str = None,
    configurable: dict


MAINMODEL_CONFIG: ModelConfig = ModelConfig(
    provider="anthropic",
    model="claude-3-5-sonnet-20240620",
    configurable={"extra_headers": {"anthropic-beta": "max-tokens-3-5-sonnet-2024-07-15"}}
)
TOOLCHECKERMODEL_CONFIG: ModelConfig = ModelConfig(
    provider="anthropic",
    model="claude-3-5-sonnet-20240620",
    configurable={"extra_headers": {"anthropic-beta": "max-tokens-3-5-sonnet-2024-07-15"}}
)
CODEEDITORMODEL_CONFIG: ModelConfig = ModelConfig(
    provider="anthropic",
    model="claude-3-5-sonnet-20240620",
    configurable={}
)
CODEEXECUTIONMODEL_CONFIG: ModelConfig = ModelConfig(
    provider="anthropic",
    model="claude-3-5-sonnet-20240620",
    configurable={}
)


In [103]:
llm = init_chat_model(
    model=MAINMODEL_CONFIG.model,
    model_provider=MAINMODEL_CONFIG.provider,
    configurable_fields="any",
)

In [104]:
from langchain_eng import (create_folder,
    create_file,
    edit_and_apply,
    execute_code,
    stop_process,
    read_file,
    read_multiple_files,
    list_files,
    tavily_search,
    get_jira_issue_details)

@tool
def shh_operation(a: int, b: int) -> int:
    """perform shh operation on 2 numbers"""    
    return a + b

In [105]:
from typing import List

tools: List[BaseTool] = [
    create_folder,
    create_file,
    edit_and_apply,
    execute_code,
    stop_process,
    read_file,
    read_multiple_files,
    list_files,
    tavily_search,
    get_jira_issue_details,
    shh_operation
]
tools_map = {t.name: t for t in tools}

In [8]:
tools[1].args

{'path': {'title': 'Path', 'default': '.', 'type': 'string'},
 'content': {'title': 'Content', 'default': '', 'type': 'string'}}

In [106]:
model = llm.bind_tools(tools)

In [126]:
configurable = MAINMODEL_CONFIG.configurable or {}
configurable.update({"model": MAINMODEL_CONFIG.model, "model_provider": MAINMODEL_CONFIG.provider,
                             "max_tokens": 8000})
messages = [
    HumanMessage(
    content="Hellow! Echo the word 'liver load🥱' also what's 8 shh 7? Additonally create a folder named 'test' in the current directory.",
)
]
response = model.invoke(
 [SystemMessage(content="You helpful assistant")] + messages,
    config={"configurable": configurable},
)

C:\Users\Rigan\AppData\Local\pypoetry\Cache\virtualenvs\turnkey-engineer-9Fj46GwB-py3.11\Lib\site-packages\langchain_core\utils\utils.py:234: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  warnings.warn(


In [108]:
response.usage_metadata["output_tokens"]

263

In [127]:
response.tool_calls

[{'name': 'execute_code',
  'args': {'code': 'print("liver load🥱")'},
  'id': 'toolu_01LXMvXKWCK3TFTsEjgdHdaW',
  'type': 'tool_call'},
 {'name': 'shh_operation',
  'args': {'a': 8, 'b': 7},
  'id': 'toolu_011yNtKxywa1HbhpN2P5ynVL',
  'type': 'tool_call'},
 {'name': 'create_folder',
  'args': {'path': 'test'},
  'id': 'toolu_01UPMgkKumAKZ7m9XJsFgLYQ',
  'type': 'tool_call'}]

In [114]:
response.content

[{'text': "Certainly! I'll address each part of your request using the available tools.\n\n1. Echoing the word 'liver load🥱':\nFor this, we can use the execute_code function to print the word.\n\n2. Calculating 8 shh 7:\nWe can use the shh_operation function for this.\n\n3. Creating a folder named 'test' in the current directory:\nWe'll use the create_folder function for this task.\n\nLet's execute these operations:",
  'type': 'text'},
 {'id': 'toolu_01DRFW4oSY8SKfNZdqbEPcsr',
  'input': {'code': 'print("liver load🥱")'},
  'name': 'execute_code',
  'type': 'tool_use'},
 {'id': 'toolu_01QJB6NnHbKaqsqxt23TtBTS',
  'input': {'a': 8, 'b': 7},
  'name': 'shh_operation',
  'type': 'tool_use'},
 {'id': 'toolu_013ZtqJMtLmGsP6LkzR1BXyC',
  'input': {'path': 'test'},
  'name': 'create_folder',
  'type': 'tool_use'}]

In [78]:
response.content

'liver load🥱\n\n'

In [81]:
messages[0].content

"Hellow! Echo the word 'liver load🥱' also what's 8 shh 7? Additonally create a folder named 'test' in the current directory."

In [85]:
from langchain_core.messages import ToolMessage

t_msg = ToolMessage(
    "hello there",
                tool_call_id=123
            )
t_msg

ToolMessage(content='hello there', tool_call_id='123')

In [98]:
t_msg.type

'tool'

In [94]:
model.__dict__

{'_default_config': {'model': 'gemini-1.5-flash',
  'model_provider': 'google_genai'},
 '_configurable_fields': 'any',
 '_config_prefix': '',
 '_queued_declarative_operations': [('bind_tools',
   ([StructuredTool(name='create_folder', description="Create a new folder at the specified path. This tool should be used when you need to create a new directory in the project structure. It will create all necessary parent directories if they don't exist. The tool will return a success message if the folder is created or already exists, and an error message if there's a problem creating the folder.\n\nArgs:\n    path: The absolute or relative path where the folder should be created. Use forward slashes (/) for path separation, even on Windows systems.\n\nReturns:\n    str: A message indicating the success or failure of the folder creation.", args_schema=<class 'pydantic.v1.main.create_folderSchema'>, func=<function create_folder at 0x000001C9AC0432E0>),
     StructuredTool(name='create_file', d

In [97]:
messages[0].type

'human'

In [119]:
dict({"1": 2})

{'1': 2}

In [120]:
response.content

[{'text': "Certainly! I'll address each part of your request using the available tools.\n\n1. Echoing the word 'liver load🥱':\nFor this, we can use the execute_code function to print the word.\n\n2. Calculating 8 shh 7:\nWe can use the shh_operation function for this.\n\n3. Creating a folder named 'test' in the current directory:\nWe'll use the create_folder function for this task.\n\nLet's execute these operations:",
  'type': 'text'},
 {'id': 'toolu_01DRFW4oSY8SKfNZdqbEPcsr',
  'input': {'code': 'print("liver load🥱")'},
  'name': 'execute_code',
  'type': 'tool_use'},
 {'id': 'toolu_01QJB6NnHbKaqsqxt23TtBTS',
  'input': {'a': 8, 'b': 7},
  'name': 'shh_operation',
  'type': 'tool_use'},
 {'id': 'toolu_013ZtqJMtLmGsP6LkzR1BXyC',
  'input': {'path': 'test'},
  'name': 'create_folder',
  'type': 'tool_use'}]

In [121]:
response.content = "hello"

In [125]:
response

AIMessage(content='hello', response_metadata={'id': 'msg_01L4TbRUHhJNVLMRCThk3o1r', 'model': 'claude-3-5-sonnet-20240620', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'input_tokens': 2758, 'output_tokens': 263}}, id='run-a5d402a6-3412-4e2a-b2bf-aa3a5e7a1183-0', usage_metadata={'input_tokens': 2758, 'output_tokens': 263, 'total_tokens': 3021})

In [123]:
response.content

'hello'

In [124]:
response.tool_calls = []